# Regularización mediante Drop-path y Cierre: Conexión con GNNs

**Módulos 4 y 5 de la presentación sobre FractalNet**

---

## Introducción

En los notebooks anteriores estudiamos la arquitectura FractalNet: cómo la **auto-similaridad estructural** permite construir redes ultra-profundas sin conexiones residuales. Vimos que una red fractal contiene múltiples rutas de distinta profundidad operando en paralelo, unidas por *Join Layers*.

Sin embargo, tener múltiples caminos en paralelo introduce un **riesgo**: la red podría aprender a depender de una sola ruta (típicamente la más profunda) mientras las demás generan ruido. Esto anularía la ventaja de la topología fractal.

En este notebook abordamos dos temas:

1. **Parte 1 (Módulo 4)**: La técnica de regularización **Drop-path**, que fuerza a cada sub-red a ser competente por sí sola.
2. **Parte 2 (Módulo 5)**: La **conexión conceptual** entre FractalNets y las Redes Neuronales de Grafos (GNNs), como cierre de la presentación.

**Dependencias**: `numpy`, `matplotlib` (no se requiere PyTorch).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# Configuración global de visualización
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'figure.dpi': 100
})

# Semilla para reproducibilidad
np.random.seed(42)

print("Librerías cargadas correctamente.")

---
# Parte 1: Regularización mediante Drop-path (Módulo 4)

## 1.1 El problema de la co-adaptación

Cuando una red tiene **múltiples caminos en paralelo** (como FractalNet), existe el riesgo de que durante el entrenamiento la red aprenda a depender casi exclusivamente de **una sola ruta** — típicamente la más profunda, que tiene mayor capacidad representacional. Las demás rutas pueden volverse "parásitas": no contribuyen información útil, o peor, generan ruido.

Este fenómeno es análogo al problema que motivó el **Dropout** clásico (Srivastava et al., 2014):

| Escala | Problema | Solución |
|--------|----------|----------|
| **Micro** (neuronas) | Co-adaptación de activaciones individuales | Dropout clásico |
| **Macro** (caminos) | Co-adaptación de rutas completas | **Drop-path** |

La filosofía es la misma — *prevenir dependencias excesivas* — pero aplicada a una escala estructural diferente.

### Visualización: Dropout vs. Drop-path

La siguiente celda muestra lado a lado cómo opera cada técnica:
- **Dropout** (izquierda): enmascara **neuronas individuales** dentro de una capa.
- **Drop-path** (derecha): enmascara **rutas completas** dentro de un bloque fractal.

Observe la diferencia de escala: micro (neuronas) vs. macro (caminos).

In [ ]:
# =============================================================================
# Visualización comparativa: Dropout clásico vs. Drop-path
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Panel izquierdo: Dropout clásico ---
ax = axes[0]
ax.set_title("Dropout clásico (escala micro)", fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 4.5)
ax.set_ylim(-0.5, 4.5)
ax.set_aspect('equal')
ax.axis('off')

# Definimos 3 capas con 5 neuronas cada una
capas = [0.5, 2.0, 3.5]  # posiciones x de cada capa
n_neuronas = 5
posiciones_y = np.linspace(0.2, 4.3, n_neuronas)

# Máscara de dropout: algunas neuronas se desactivan (probabilidad ~30%)
np.random.seed(42)
mascara_dropout = np.random.rand(len(capas), n_neuronas) > 0.3

# Dibujamos neuronas y conexiones
for i, x in enumerate(capas):
    for j, y in enumerate(posiciones_y):
        activa = mascara_dropout[i, j]
        color = '#2196F3' if activa else '#FFCDD2'      # azul=activa, rojo claro=inactiva
        borde = '#1565C0' if activa else '#E53935'
        estilo = '-' if activa else '--'
        ax.add_patch(plt.Circle((x, y), 0.18, fc=color, ec=borde,
                                lw=2, linestyle=estilo, zorder=3))
        if not activa:
            ax.plot([x - 0.12, x + 0.12], [y - 0.12, y + 0.12],
                    color='#E53935', lw=2, zorder=4)
            ax.plot([x - 0.12, x + 0.12], [y + 0.12, y - 0.12],
                    color='#E53935', lw=2, zorder=4)

    # Conexiones a la capa siguiente
    if i < len(capas) - 1:
        x_next = capas[i + 1]
        for j1, y1 in enumerate(posiciones_y):
            for j2, y2 in enumerate(posiciones_y):
                if mascara_dropout[i, j1] and mascara_dropout[i + 1, j2]:
                    ax.plot([x + 0.18, x_next - 0.18], [y1, y2],
                            color='#90CAF9', lw=0.6, alpha=0.5, zorder=1)

    # Etiqueta de capa
    ax.text(x, -0.3, f"Capa {i+1}", ha='center', fontsize=10)

ax.text(2.0, 4.8, "Se enmascaran neuronas individuales",
        ha='center', fontsize=10, style='italic', color='#555')

# Leyenda del panel izquierdo
leg_activa = mpatches.Patch(fc='#2196F3', ec='#1565C0', label='Neurona activa')
leg_inactiva = mpatches.Patch(fc='#FFCDD2', ec='#E53935', label='Neurona desactivada')
ax.legend(handles=[leg_activa, leg_inactiva], loc='lower center',
          fontsize=9, ncol=2, framealpha=0.9)


# --- Panel derecho: Drop-path ---
ax = axes[1]
ax.set_title("Drop-path (escala macro)", fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 5.0)
ax.set_aspect('equal')
ax.axis('off')

# Definimos 4 rutas paralelas (columnas de un bloque fractal)
n_rutas = 4
colores_ruta = ['#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
colores_ruta_off = ['#C8E6C9', '#FFE0B2', '#E1BEE7', '#B2EBF2']
nombres_ruta = ['Col 1\n(1 capa)', 'Col 2\n(2 capas)', 'Col 3\n(4 capas)', 'Col 4\n(8 capas)']

# Simulamos drop-path: ruta 1 y 3 activas; ruta 0 y 2 desactivadas
rutas_activas = [False, True, False, True]

# Posiciones
y_entrada = 0.2
y_salida = 4.3
x_posiciones = np.linspace(0.5, 4.5, n_rutas)

# Nodo de entrada
ax.add_patch(plt.Circle((2.5, y_entrada), 0.22, fc='#FFF9C4', ec='#F9A825',
                         lw=2, zorder=3))
ax.text(2.5, y_entrada, "x", ha='center', va='center', fontsize=12, fontweight='bold')

# Nodo de salida (Join)
ax.add_patch(FancyBboxPatch((1.7, y_salida - 0.15), 1.6, 0.4,
             boxstyle="round,pad=0.1", fc='#E3F2FD', ec='#1565C0', lw=2, zorder=3))
ax.text(2.5, y_salida + 0.05, "Join (promedio)", ha='center', va='center',
        fontsize=10, fontweight='bold')

# Dibujamos cada ruta
for k in range(n_rutas):
    activa = rutas_activas[k]
    x = x_posiciones[k]
    color = colores_ruta[k] if activa else colores_ruta_off[k]
    alpha = 1.0 if activa else 0.35
    lw = 3 if activa else 1.5
    estilo = '-' if activa else '--'

    # Profundidad de la ruta (número de capas internas)
    n_capas_ruta = 2 ** k
    y_capas = np.linspace(y_entrada + 0.6, y_salida - 0.5, min(n_capas_ruta, 4))

    # Línea de entrada al primer nodo de la ruta
    ax.annotate("", xy=(x, y_capas[0] - 0.12), xytext=(2.5, y_entrada + 0.22),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                alpha=alpha, linestyle=estilo))

    # Nodos de la ruta
    for c, yc in enumerate(y_capas):
        ax.add_patch(plt.Circle((x, yc), 0.12, fc=color, ec='black',
                                lw=1, alpha=alpha, zorder=3))

    # Conexiones internas de la ruta
    for c in range(len(y_capas) - 1):
        ax.plot([x, x], [y_capas[c] + 0.12, y_capas[c + 1] - 0.12],
                color=color, lw=lw, alpha=alpha, linestyle=estilo, zorder=2)

    # Línea del último nodo al Join
    ax.annotate("", xy=(2.5, y_salida - 0.15), xytext=(x, y_capas[-1] + 0.12),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                alpha=alpha, linestyle=estilo))

    # Etiqueta de ruta
    etiqueta = nombres_ruta[k]
    estado = "ACTIVA" if activa else "DROPPED"
    ax.text(x, y_salida + 0.65, etiqueta, ha='center', va='bottom', fontsize=8,
            color=color if activa else '#999')
    ax.text(x, y_salida + 0.55, estado, ha='center', va='top', fontsize=8,
            fontweight='bold', color=colores_ruta[k] if activa else '#E53935')

ax.text(2.5, -0.4, "Se enmascaran rutas completas (sub-redes enteras)",
        ha='center', fontsize=10, style='italic', color='#555')

plt.suptitle("Comparación: Dropout clásico vs. Drop-path",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 1.2 Mecanismo Drop-path

Drop-path es una extensión conceptual de Dropout que opera a nivel de **rutas completas** en lugar de neuronas individuales. Existen dos variantes:

### Drop-path Local
En cada **Join Layer**, cada entrada $x_i$ se elimina de forma independiente con probabilidad fija $p = 0.15$ (valor usado en los experimentos del paper). La restricción es que **al menos una ruta debe sobrevivir** para que la información pueda fluir.

### Drop-path Global
Se usa un **modelo de mezcla** por mini-batch:
- Con probabilidad **50%** se aplica muestreo **local** (como arriba).
- Con probabilidad **50%** se aplica muestreo **global**: la red entera se restringe a **una sola columna** seleccionada aleatoriamente.

El muestreo global promueve que cada columna individual sea un **predictor competente** por sí mismo, ya que periódicamente debe "cargar" con toda la responsabilidad de la predicción.

### Implementación del bloque fractal con Drop-path

En la siguiente celda implementamos un bloque fractal simplificado con $C = 4$ columnas que soporta ambos modos de Drop-path. Usamos transformaciones lineales simples (matrices aleatorias + ReLU) para simular las capas de procesamiento.

In [ ]:
# =============================================================================
# Implementación de un Bloque Fractal con Drop-path
# =============================================================================

def relu(x):
    """Función de activación ReLU elemento a elemento."""
    return np.maximum(0, x)


class CapaLineal:
    """Capa lineal simple: y = ReLU(Wx + b)
    
    Simula una capa de procesamiento (en FractalNet real sería convolucional).
    """
    def __init__(self, dim_entrada, dim_salida):
        # Inicialización He (adecuada para ReLU)
        self.W = np.random.randn(dim_salida, dim_entrada) * np.sqrt(2.0 / dim_entrada)
        self.b = np.zeros(dim_salida)
    
    def __call__(self, x):
        return relu(self.W @ x + self.b)


class BloqueFractal:
    """Bloque fractal con C columnas y soporte para Drop-path.
    
    Cada columna k tiene 2^k capas de procesamiento (profundidad exponencial).
    El Join Layer promedia las salidas de las columnas activas.
    
    Parámetros:
        C: número de columnas (default=4)
        dim: dimensionalidad de las señales
        p_drop_local: probabilidad de desactivar cada ruta en modo local (default=0.15)
        p_global: probabilidad de usar muestreo global vs local (default=0.5)
    """
    def __init__(self, C=4, dim=8, p_drop_local=0.15, p_global=0.5):
        self.C = C
        self.dim = dim
        self.p_drop_local = p_drop_local
        self.p_global = p_global
        
        # Crear las capas para cada columna
        # Columna k tiene 2^k capas
        self.columnas = []
        for k in range(C):
            n_capas = 2 ** k
            capas = [CapaLineal(dim, dim) for _ in range(n_capas)]
            self.columnas.append(capas)
    
    def _muestreo_local(self):
        """Drop-path local: cada ruta se desactiva con probabilidad p_drop_local.
        Garantiza que al menos una ruta sobreviva."""
        máscara = np.random.rand(self.C) > self.p_drop_local
        # Garantizar al menos una ruta activa
        if not máscara.any():
            idx = np.random.randint(self.C)
            máscara[idx] = True
        return máscara
    
    def _muestreo_global(self):
        """Drop-path global: se selecciona una sola columna."""
        máscara = np.zeros(self.C, dtype=bool)
        idx = np.random.randint(self.C)
        máscara[idx] = True
        return máscara
    
    def generar_mascara(self):
        """Modelo de mezcla: 50% local, 50% global."""
        if np.random.rand() < self.p_global:
            modo = "global"
            máscara = self._muestreo_global()
        else:
            modo = "local"
            máscara = self._muestreo_local()
        return máscara, modo
    
    def forward(self, x, máscara=None, modo_inferencia=False):
        """Propagación hacia adelante por el bloque fractal.
        
        Args:
            x: vector de entrada (dim,)
            máscara: máscara booleana opcional (si None, se genera automáticamente)
            modo_inferencia: si True, usa todas las columnas (sin drop-path)
        
        Returns:
            salida: vector de salida (dim,)
            máscara: máscara usada
            modo: 'local', 'global' o 'inferencia'
        """
        if modo_inferencia:
            máscara = np.ones(self.C, dtype=bool)
            modo = "inferencia"
        elif máscara is None:
            máscara, modo = self.generar_mascara()
        else:
            modo = "manual"
        
        # Procesar cada columna activa
        salidas = []
        for k in range(self.C):
            if máscara[k]:
                h = x.copy()
                for capa in self.columnas[k]:
                    h = capa(h)
                salidas.append(h)
        
        # Join Layer: promedio de las salidas activas
        salida = np.mean(salidas, axis=0)
        return salida, máscara, modo
    
    def info_columnas(self):
        """Muestra información sobre la profundidad de cada columna."""
        for k in range(self.C):
            n_capas = 2 ** k
            print(f"  Columna {k+1}: {n_capas} capa(s) — profundidad {'baja' if k == 0 else 'media' if k < self.C-1 else 'alta'}")


# Crear el bloque fractal
np.random.seed(123)
bloque = BloqueFractal(C=4, dim=8)

print("Bloque Fractal creado con C=4 columnas:")
bloque.info_columnas()
print(f"\nParámetros de drop-path:")
print(f"  p_drop_local = {bloque.p_drop_local}")
print(f"  p_global = {bloque.p_global}")

### Procesando datos con distintas máscaras de Drop-path

Veamos cómo el mismo dato de entrada produce **diferentes caminos activos** en cada pasada. Esto simula lo que ocurre durante el entrenamiento: en cada mini-batch, el bloque fractal elige una configuración distinta de rutas activas.

In [ ]:
# =============================================================================
# Demostración: múltiples pasadas con diferentes caminos activos
# =============================================================================

# Dato de entrada fijo
x_entrada = np.random.randn(8) * 0.5 + 1.0

print("Vector de entrada x:")
print(f"  {np.round(x_entrada, 3)}\n")
print("=" * 65)

# Realizamos 8 pasadas mostrando qué rutas se activan cada vez
np.random.seed(77)
n_pasadas = 8
resultados = []

for i in range(n_pasadas):
    salida, máscara, modo = bloque.forward(x_entrada)
    resultados.append((salida, máscara, modo))
    
    rutas_str = ", ".join([f"Col {k+1}" for k in range(4) if máscara[k]])
    print(f"\nPasada {i+1} [{modo.upper():>7s}]:")
    print(f"  Rutas activas: {rutas_str}")
    print(f"  Máscara:       {máscara.astype(int)}")
    print(f"  ||salida||:    {np.linalg.norm(salida):.4f}")

## 1.3 Simulación del modelo de mezcla

Ahora ejecutamos **muchas iteraciones** (1000) del modelo de mezcla para observar las estadísticas de activación. Según el diseño del paper:
- ~50% de los mini-batches usan muestreo **local** (múltiples rutas activas).
- ~50% usan muestreo **global** (una sola columna activa).

Visualizaremos:
1. La proporción de veces que se usa cada modo (local vs. global).
2. La frecuencia de activación de cada columna.
3. Ejemplos de cuáles columnas están activas en una muestra de iteraciones.

In [ ]:
# =============================================================================
# Simulación del modelo de mezcla: 1000 iteraciones
# =============================================================================

np.random.seed(42)
n_iteraciones = 1000

# Registros para las estadísticas
modos_registro = []          # 'local' o 'global' por iteración
mascaras_registro = []       # máscara booleana (C,) por iteración

for _ in range(n_iteraciones):
    máscara, modo = bloque.generar_mascara()
    modos_registro.append(modo)
    mascaras_registro.append(máscara)

mascaras_registro = np.array(mascaras_registro)  # (n_iteraciones, C)
modos_registro = np.array(modos_registro)

# --- Estadísticas ---
n_local = np.sum(modos_registro == "local")
n_global = np.sum(modos_registro == "global")

print(f"Resultados de {n_iteraciones} iteraciones:")
print(f"  Modo local:  {n_local} ({100*n_local/n_iteraciones:.1f}%)")
print(f"  Modo global: {n_global} ({100*n_global/n_iteraciones:.1f}%)")
print()

# Frecuencia de activación por columna
freq_activacion = mascaras_registro.mean(axis=0)
for k in range(bloque.C):
    print(f"  Columna {k+1}: activa en {100*freq_activacion[k]:.1f}% de las iteraciones")

# --- Visualización ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Proporción local vs global
ax = axes[0]
etiquetas_modo = ['Local\n(múltiples rutas)', 'Global\n(una sola columna)']
valores_modo = [n_local, n_global]
colores_modo = ['#42A5F5', '#FF7043']
barras = ax.bar(etiquetas_modo, valores_modo, color=colores_modo, edgecolor='black', lw=1.2)
for barra, val in zip(barras, valores_modo):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 15,
            f"{val}\n({100*val/n_iteraciones:.1f}%)", ha='center', fontsize=11)
ax.set_ylabel("Número de iteraciones")
ax.set_title("Distribución de modos de muestreo", fontweight='bold')
ax.set_ylim(0, max(valores_modo) * 1.2)

# Panel 2: Frecuencia de activación por columna
ax = axes[1]
colores_col = ['#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
nombres_col = [f"Col {k+1}\n({2**k} capas)" for k in range(bloque.C)]
barras = ax.bar(nombres_col, freq_activacion * 100, color=colores_col,
                edgecolor='black', lw=1.2)
for barra, freq in zip(barras, freq_activacion):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 1.5,
            f"{100*freq:.1f}%", ha='center', fontsize=11)
ax.set_ylabel("% de iteraciones activa")
ax.set_title("Frecuencia de activación por columna", fontweight='bold')
ax.set_ylim(0, 110)
ax.axhline(y=85, color='gray', linestyle='--', alpha=0.5, label='Esperado local (~85%)')
ax.legend(fontsize=9)

# Panel 3: Mapa de calor de las primeras 50 iteraciones
ax = axes[2]
n_mostrar = 50
im = ax.imshow(mascaras_registro[:n_mostrar].T, aspect='auto', cmap='YlGn',
               interpolation='nearest')
ax.set_yticks(range(bloque.C))
ax.set_yticklabels([f"Col {k+1}" for k in range(bloque.C)])
ax.set_xlabel("Iteración")
ax.set_title(f"Rutas activas (primeras {n_mostrar} iteraciones)", fontweight='bold')

# Marcar iteraciones globales con un punto rojo
for i in range(n_mostrar):
    if modos_registro[i] == "global":
        ax.plot(i, -0.5, 'rv', markersize=5, clip_on=False)

ax.text(n_mostrar/2, -1.1, "Triángulos rojos = modo global",
        ha='center', fontsize=9, color='red', style='italic')

plt.colorbar(im, ax=ax, shrink=0.8, label="Activa (1) / Inactiva (0)")
plt.tight_layout()
plt.show()

## 1.4 Extracción de sub-redes: la propiedad "anytime"

Una consecuencia directa del entrenamiento con Drop-path (especialmente el modo global) es que **cada columna individual** se entrena para ser un predictor razonable por sí sola. Esto habilita la propiedad **"anytime"**: en inferencia, podemos extraer sub-redes de distinta profundidad (y por lo tanto distinta velocidad) sin reentrenar.

- Necesitamos **velocidad**: usamos solo la Columna 1 (1 capa, la más rápida).
- Necesitamos **precisión**: usamos todas las columnas (o solo la más profunda).
- Queremos un **balance**: usamos un subconjunto intermedio.

En la siguiente celda demostramos que cada columna individual produce salidas razonables (no degeneradas), lo cual no ocurriria si una columna dominara durante el entrenamiento y las demas se "atrofiaran".

In [ ]:
# =============================================================================
# Demostración de extracción de sub-redes (propiedad "anytime")
# =============================================================================

np.random.seed(42)

# Generamos un conjunto de datos de prueba
n_muestras = 200
datos_prueba = np.random.randn(n_muestras, 8) * 0.5 + 1.0

# Procesamos con cada columna individual y con todas las columnas
salidas_por_columna = {}   # {nombre: array de normas de salidas}

# Cada columna individual
for k in range(bloque.C):
    máscara = np.zeros(bloque.C, dtype=bool)
    máscara[k] = True
    normas = []
    for x in datos_prueba:
        salida, _, _ = bloque.forward(x, máscara=máscara)
        normas.append(np.linalg.norm(salida))
    salidas_por_columna[f"Col {k+1} ({2**k} capas)"] = normas

# Todas las columnas (inferencia completa)
normas_full = []
for x in datos_prueba:
    salida, _, _ = bloque.forward(x, modo_inferencia=True)
    normas_full.append(np.linalg.norm(salida))
salidas_por_columna["Todas (inferencia)"] = normas_full

# --- Visualización ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Distribución de normas de salida por configuración
ax = axes[0]
colores = ['#4CAF50', '#FF9800', '#9C27B0', '#00BCD4', '#F44336']
posiciones = []
etiquetas = []

datos_violin = []
for i, (nombre, normas) in enumerate(salidas_por_columna.items()):
    datos_violin.append(normas)
    posiciones.append(i)
    etiquetas.append(nombre)

vp = ax.violinplot(datos_violin, positions=posiciones, showmeans=True, showmedians=True)
for i, body in enumerate(vp['bodies']):
    body.set_facecolor(colores[i])
    body.set_alpha(0.7)
vp['cmeans'].set_color('black')
vp['cmedians'].set_color('white')

ax.set_xticks(posiciones)
ax.set_xticklabels(etiquetas, fontsize=9, rotation=15)
ax.set_ylabel("||salida||  (norma L2)")
ax.set_title("Distribución de salidas por sub-red extraída", fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Panel 2: Comparación de salidas para una muestra específica
ax = axes[1]
idx_muestra = 0
x_muestra = datos_prueba[idx_muestra]

salidas_individuales = []
etiquetas_bar = []

for k in range(bloque.C):
    máscara = np.zeros(bloque.C, dtype=bool)
    máscara[k] = True
    salida, _, _ = bloque.forward(x_muestra, máscara=máscara)
    salidas_individuales.append(salida)
    etiquetas_bar.append(f"Col {k+1}")

# Salida completa
salida_full, _, _ = bloque.forward(x_muestra, modo_inferencia=True)
salidas_individuales.append(salida_full)
etiquetas_bar.append("Todas")

# Mostramos las primeras 4 dimensiones del vector de salida
n_dims_mostrar = 4
x_pos = np.arange(n_dims_mostrar)
ancho = 0.15

for i, (salida, etiqueta) in enumerate(zip(salidas_individuales, etiquetas_bar)):
    offset = (i - len(salidas_individuales)/2) * ancho
    ax.bar(x_pos + offset, salida[:n_dims_mostrar], width=ancho,
           label=etiqueta, color=colores[i], edgecolor='black', lw=0.5, alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels([f"Dim {d+1}" for d in range(n_dims_mostrar)])
ax.set_ylabel("Valor de salida")
ax.set_title("Salida de cada sub-red para una muestra fija\n(primeras 4 dimensiones)",
             fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Resumen numérico
print("\nResumen: norma media de salida por configuración")
print("-" * 50)
for nombre, normas in salidas_por_columna.items():
    print(f"  {nombre:25s}:  media={np.mean(normas):.4f},  std={np.std(normas):.4f}")
print()
print("Observación: Todas las columnas producen salidas no degeneradas (norma > 0)")
print("y con variabilidad razonable. Esto confirma que cada sub-red es funcional.")

---
# Parte 2: Cierre — Conexión con GNNs (Módulo 5)

## 2.1 Recapitulación

A lo largo de estos notebooks hemos recorrido el siguiente camino:

1. **Notebook 01**: Las GNNs procesan datos no euclidianos mediante **paso de mensajes** — cada nodo agrega información de sus vecinos.
2. **Notebook 02**: Al apilar muchas capas de paso de mensajes, las representaciones de los nodos **convergen** y pierden discriminabilidad (**over-smoothing**). Las conexiones residuales alivian el problema pero no lo resuelven completamente.
3. **Notebook 03**: Las **FractalNets** (Larsson et al., 2017) demuestran que se pueden construir redes ultra-profundas sin conexiones residuales, usando una topología de rutas paralelas de distinta profundidad unidas por *Join Layers*.
4. **Notebook 04** (este): El **Drop-path** fuerza a cada sub-red fractal a ser competente por sí sola, habilitando la propiedad "anytime".

### La idea clave de FractalNets

> La contribución fundamental no es una arquitectura específica, sino un **principio de diseño**: la **autosimilitud estructural** como alternativa a las conexiones residuales para habilitar profundidad.

Esto fue desarrollado para **CNNs** (clasificación de imágenes), pero el problema de fondo — la dificultad de entrenar redes muy profundas — es compartido con las GNNs.

## 2.2 Pregunta abierta: ¿Cómo sería una GNN fractal?

¿Podemos transferir la topología fractal al paradigma de paso de mensajes en grafos? La analogía directa sería:

| FractalNet (CNNs) | Hipotética FractalGNN |
|---|---|
| Capa convolucional | Capa de agregación de vecinos |
| Ruta profunda ($2^{C-1}$ capas) | Agregación de $K$-saltos (*hops*) |
| Ruta superficial (1 capa) | Agregación de vecinos directos (1 salto) |
| Join Layer (promedio) | Promedio de representaciones de distinta profundidad |
| **Sin señal de identidad privilegiada** | **Ningún nivel de "hop" es primario** |

La siguiente celda visualiza esta idea conceptualmente.

In [ ]:
# =============================================================================
# Diagrama conceptual: FractalGNN hipotética
# =============================================================================
# Mostramos cómo la topología fractal se traduciría al dominio de grafos:
# - En lugar de convoluciones, usamos capas de agregación de vecinos
# - Cada columna representa un número distinto de "saltos" (hops)
# - El Join Layer combina información de distintas escalas de vecindario

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Panel izquierdo: FractalNet original (CNNs) ---
ax = axes[0]
ax.set_title("FractalNet original (CNNs)", fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-1.0, 6.5)
ax.set_aspect('equal')
ax.axis('off')

# Colores por columna
colores = ['#4CAF50', '#FF9800', '#9C27B0']
nombres_col = ['Col 1\n(1 conv)', 'Col 2\n(2 conv)', 'Col 3\n(4 conv)']
profundidades = [1, 2, 4]
x_cols = [1.0, 2.5, 4.0]

# Entrada
ax.add_patch(FancyBboxPatch((1.5, 5.7), 1.5, 0.5,
             boxstyle="round,pad=0.1", fc='#FFF9C4', ec='#F9A825', lw=2, zorder=3))
ax.text(2.25, 5.95, "Entrada (imagen)", ha='center', va='center', fontsize=10, fontweight='bold')

# Columnas
for i, (x, prof, color, nombre) in enumerate(zip(x_cols, profundidades, colores, nombres_col)):
    y_capas = np.linspace(5.0, 1.5, prof + 2)[1:-1]  # posiciones de las capas
    
    # Conexión desde entrada
    ax.annotate("", xy=(x, y_capas[0] + 0.2), xytext=(2.25, 5.7),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    
    # Capas convolucionales
    for y in y_capas:
        ax.add_patch(FancyBboxPatch((x - 0.35, y - 0.15), 0.7, 0.3,
                     boxstyle="round,pad=0.05", fc=color, ec='black', lw=1, alpha=0.8, zorder=3))
        ax.text(x, y, "Conv", ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    
    # Conexiones entre capas
    for j in range(len(y_capas) - 1):
        ax.plot([x, x], [y_capas[j] - 0.15, y_capas[j+1] + 0.15],
                color=color, lw=2)
    
    # Conexión al Join
    ax.annotate("", xy=(2.25, 0.85), xytext=(x, y_capas[-1] - 0.15),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    
    # Etiqueta
    ax.text(x, 6.3, nombre, ha='center', fontsize=9, color=color, fontweight='bold')

# Join Layer
ax.add_patch(FancyBboxPatch((1.2, 0.5), 2.1, 0.5,
             boxstyle="round,pad=0.1", fc='#E3F2FD', ec='#1565C0', lw=2, zorder=3))
ax.text(2.25, 0.75, "Join (promedio)", ha='center', va='center', fontsize=10, fontweight='bold')

# Salida
ax.annotate("", xy=(2.25, -0.1), xytext=(2.25, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(2.25, -0.4, "Salida", ha='center', fontsize=10, fontweight='bold')


# --- Panel derecho: FractalGNN hipotética ---
ax = axes[1]
ax.set_title("FractalGNN hipotética (Grafos)", fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 5.0)
ax.set_ylim(-1.0, 6.5)
ax.set_aspect('equal')
ax.axis('off')

nombres_col_gnn = ['Col 1\n(1-hop)', 'Col 2\n(2-hop)', 'Col 3\n(4-hop)']
etiquetas_capas = [['AGG\n1-hop'], ['AGG\n1-hop', 'AGG\n1-hop'],
                   ['AGG\n1-hop', 'AGG\n1-hop', 'AGG\n1-hop', 'AGG\n1-hop']]

# Entrada
ax.add_patch(FancyBboxPatch((1.5, 5.7), 1.5, 0.5,
             boxstyle="round,pad=0.1", fc='#FFF9C4', ec='#F9A825', lw=2, zorder=3))
ax.text(2.25, 5.95, "Nodo v (features)", ha='center', va='center', fontsize=10, fontweight='bold')

# Columnas
for i, (x, prof, color, nombre) in enumerate(zip(x_cols, profundidades, colores, nombres_col_gnn)):
    y_capas = np.linspace(5.0, 1.5, prof + 2)[1:-1]
    
    ax.annotate("", xy=(x, y_capas[0] + 0.2), xytext=(2.25, 5.7),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    
    for j, y in enumerate(y_capas):
        ax.add_patch(FancyBboxPatch((x - 0.4, y - 0.18), 0.8, 0.36,
                     boxstyle="round,pad=0.05", fc=color, ec='black', lw=1, alpha=0.8, zorder=3))
        ax.text(x, y, etiquetas_capas[i][j], ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
    
    for j in range(len(y_capas) - 1):
        ax.plot([x, x], [y_capas[j] - 0.18, y_capas[j+1] + 0.18],
                color=color, lw=2)
    
    ax.annotate("", xy=(2.25, 0.85), xytext=(x, y_capas[-1] - 0.18),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    
    ax.text(x, 6.3, nombre, ha='center', fontsize=9, color=color, fontweight='bold')

# Join Layer
ax.add_patch(FancyBboxPatch((1.2, 0.5), 2.1, 0.5,
             boxstyle="round,pad=0.1", fc='#E3F2FD', ec='#1565C0', lw=2, zorder=3))
ax.text(2.25, 0.75, "Join (promedio)", ha='center', va='center', fontsize=10, fontweight='bold')

# Salida
ax.annotate("", xy=(2.25, -0.1), xytext=(2.25, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(2.25, -0.4, "h_v actualizado", ha='center', fontsize=10, fontweight='bold')

# Anotación sobre over-smoothing
ax.annotate("Vecinos directos\n(no se suaviza)",
            xy=(1.0, 3.25), xytext=(-0.3, 4.5),
            fontsize=8, color='#4CAF50', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=1.5))

ax.annotate("Vecinos lejanos\n(riesgo de suavizado)",
            xy=(4.0, 3.25), xytext=(4.3, 4.8),
            fontsize=8, color='#9C27B0', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=1.5))

ax.text(2.25, -0.8, "El Join promedia AMBAS escalas sin privilegiar\nninguna → podría aliviar el over-smoothing (hipótesis)",
        ha='center', fontsize=9, style='italic', color='#555',
        bbox=dict(boxstyle='round,pad=0.3', fc='#FFF3E0', ec='#FF9800', alpha=0.8))

plt.suptitle("Analogía: de FractalNet (CNNs) a FractalGNN (Grafos)",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 2.3 ¿Por qué esto *podría* mitigar el over-smoothing?

Recordemos el problema del Notebook 02: en una GNN convencional, al apilar $k$ capas de agregación, los nodos terminan integrando información de todo el grafo y sus representaciones se alinean, perdiendo discriminabilidad.

La **hipótesis** es que una GNN fractal podría aliviar este problema por las siguientes razones (no demostradas, sino intuiciones de diseño):

1. **La ruta superficial (1-hop) siempre está presente**: el Join Layer siempre recibiría una representación basada solo en los vecinos directos, que preserva la información local.
2. **La ruta profunda (K-hops) aporta contexto global**, pero su contribución se **promediaría** con la de las rutas más superficiales, diluyendo potencialmente el efecto de suavizado.
3. **No hay señal privilegiada**: a diferencia de las skip connections (donde la identidad tiene estatus especial), el Join trataría todas las escalas por igual. Ningún nivel de profundidad dominaría por diseño.
4. **Drop-path reforzaría la robustez**: al entrenar cada columna para funcionar independientemente, la red no dependería exclusivamente de la ruta más profunda (que es la más propensa al over-smoothing).

**Importante:** Esta es una **dirección de investigación abierta**, no un resultado establecido. No existen (hasta donde sabemos) publicaciones que implementen y evalúen una FractalGNN. El valor de presentar esta idea no es como solución probada, sino como ejemplo de cómo un principio de diseño arquitectónico (la autosimilitud) puede inspirar nuevas líneas de trabajo.

---

## Bibliografía

1. **Larsson, G., Maire, M., & Shakhnarovich, G.** (2017). *FractalNet: Ultra-Deep Neural Networks without Residuals*. ICLR 2017.
2. **Hamilton, W., Ying, Z., & Leskovec, J.** (2017). *Inductive Representation Learning on Large Graphs (GraphSAGE)*. NIPS.
3. **Li, Q., Han, Z., & Wu, X. M.** (2018). *Deeper Insights into Graph Convolutional Networks for Semi-Supervised Learning*. AAAI.
4. **Aggarwal, C. C.** (2023). *Neural Networks and Deep Learning* (2nd ed.). Springer.
5. **Hinton, G. E., et al.** (2012). *Improving neural networks by preventing co-adaptation of feature detectors*. arXiv:1207.0580.
6. **Srivastava, N., et al.** (2014). *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR.
7. **He, K., Zhang, X., Ren, S., & Sun, J.** (2016). *Deep Residual Learning for Image Recognition*. CVPR.